In [1]:
%pwd

'c:\\Users\\Amal\\Desktop\\End to End project Mlops mlflow github\\End-to-End-project-Mlops-Mlflow\\research'

In [2]:
%cd ..

c:\Users\Amal\Desktop\End to End project Mlops mlflow github\End-to-End-project-Mlops-Mlflow


C:\Users\Amal\AppData\Roaming\Python\Python39\site-packages\IPython\core\magics\osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
%pwd

'c:\\Users\\Amal\\Desktop\\End to End project Mlops mlflow github\\End-to-End-project-Mlops-Mlflow'

Update the entity

In [4]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True) # frozen=True signifie que les instances de la dataclass devient immutable (en lecture seule après sa création, modification des instance non autorisé)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

Update the configuration manager

In [5]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root]) # crée le dossier artifacts (correspond au artifacts_root dans fichier config.yaml)


    # Cette fonction retourne un objet de type DataIngestionConfig
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config

Update the components

In [8]:
import os
import urllib.request as request
import zipfile
from mlProject import logger
from mlProject.utils.common import get_size

In [24]:
import urllib.request

In [28]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config


    # download data
    def download_file(self):
        
     #   if not os.path.exists(self.config.local_data_file):
         
            os.makedirs(os.path.dirname(self.config.local_data_file), exist_ok=True)

            urllib.request.urlretrieve(self.config.source_URL, self.config.local_data_file)

            logger.info(f"{self.config.source_URL} download! with following info: \n{self.config.local_data_file}")
        
       # else:
           # logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")
        


    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

Update the pipline

In [29]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

# => Results: Data is downloaded and unzipped in artifacts/data_ingestion.

[2026-06-24 13:04:20,030: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-24 13:04:20,035: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-24 13:04:20,043: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-06-24 13:04:20,048: INFO: common: created directory at: artifacts]
[2026-06-24 13:04:20,053: INFO: common: created directory at: artifacts/data_ingestion]
[2026-06-24 13:04:20,503: INFO: 1276061174: https://raw.githubusercontent.com/Amal1703/End-to-End-project-Mlops-Mlflow/main/dataset/Housing.csv download! with following info: 
artifacts/data_ingestion/data.zip]


BadZipFile: File is not a zip file